In [31]:
import os
import cv2
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import (classification_report,
                             confusion_matrix,
                             ConfusionMatrixDisplay)
 
# ---------- Đường dẫn Kaggle ----------
DATA_ROOT   = Path("/kaggle/input/datasets/minhquang2701/food101-15cl")
# IMAGE_DIR   = DATA_ROOT / "images"
IMAGE_DIR   = DATA_ROOT
OUTPUT_DIR  = Path("/kaggle/working/portion_output")
OUTPUT_DIR.mkdir(exist_ok=True)
 
# ---------- 15 class giống giữa kỳ ----------
CLASSES = [
    "caesar_salad", "chocolate_cake", "donuts", "dumplings",
    "french_fries", "fried_rice", "grilled_salmon", "hamburger",
    "hot_dog", "ice_cream", "omelette", "pancakes",
    "pizza", "ramen", "sushi",
]
 
# ---------- Ngưỡng Portion (có thể chỉnh) ----------
# Dựa trên area_ratio (diện tích thức ăn / tổng diện tích ảnh)
# Kết hợp với weighted_score từ nhiều ratio
THRESHOLDS = {
    "small":  0.28,   # weighted_score < 0.28  → Small
    "large":  0.55,   # weighted_score > 0.55  → Large
}                     # còn lại                → Medium
 
LABEL_MAP = {0: "Small", 1: "Medium", 2: "Large"}
COLOR_MAP  = {"Small": "#3498db", "Medium": "#f39c12", "Large": "#e74c3c"}
 
SEED = 42
N_SAMPLE_PER_CLASS = 100   # số ảnh mỗi class để chạy pipeline

**SEGMENTATION**

In [32]:
def preprocess_image(img_bgr: np.ndarray,
                     size: int = 512) -> np.ndarray:
    """
    Bilateral filter + CLAHE (dùng lại từ giữa kỳ) + resize.
    Giữ nguyên tỷ lệ, pad reflect.
    """
    filtered = cv2.bilateralFilter(img_bgr, d=5,
                                   sigmaColor=50, sigmaSpace=50)
    lab = cv2.cvtColor(filtered, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
 
    h, w = enhanced.shape[:2]
    scale = size / max(h, w)
    nh, nw = int(h * scale), int(w * scale)
    resized = cv2.resize(enhanced, (nw, nh),
                         interpolation=cv2.INTER_LINEAR)
    # Pad về size x size
    pad_h = size - nh
    pad_w = size - nw
    padded = cv2.copyMakeBorder(
        resized,
        pad_h // 2, pad_h - pad_h // 2,
        pad_w // 2, pad_w - pad_w // 2,
        cv2.BORDER_REFLECT,
    )
    return padded

In [33]:
def build_initial_mask_hsv_lab(img_bgr: np.ndarray) -> np.ndarray:
    """
    Tạo mask ban đầu từ HSV + Lab Otsu (dùng lại từ giữa kỳ).
    Output: binary mask uint8 (0 = background, 255 = foreground)
    """
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
 
    # Kênh S (HSV)
    _, mask_s = cv2.threshold(
        hsv[:, :, 1], 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    # Kênh b (Lab)
    _, mask_b = cv2.threshold(
        lab[:, :, 2], 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
 
    combined = cv2.bitwise_or(mask_s, mask_b)
    # Morphology closing để nối vùng bị đứt
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    combined = cv2.morphologyEx(combined, cv2.MORPH_CLOSE, kernel)
    combined = cv2.morphologyEx(combined, cv2.MORPH_OPEN,
                                cv2.getStructuringElement(
                                    cv2.MORPH_ELLIPSE, (5, 5)))
    return combined


In [34]:
def grabcut_segment(img_bgr: np.ndarray,
                    n_iter: int = 5) -> tuple[np.ndarray, np.ndarray]:
    """
    GrabCut với initial mask từ HSV+Lab.
    Trả về: (binary_mask, masked_image)
    """
    h, w = img_bgr.shape[:2]
    init_mask = build_initial_mask_hsv_lab(img_bgr)
 
    # Chuyển sang định dạng GrabCut
    gc_mask = np.where(init_mask > 0,
                       cv2.GC_PR_FGD,
                       cv2.GC_PR_BGD).astype(np.uint8)
 
    # Đảm bảo viền là background chắc chắn
    border = max(5, int(min(h, w) * 0.03))
    gc_mask[:border, :]  = cv2.GC_BGD
    gc_mask[-border:, :] = cv2.GC_BGD
    gc_mask[:, :border]  = cv2.GC_BGD
    gc_mask[:, -border:] = cv2.GC_BGD
 
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)
 
    try:
        cv2.grabCut(img_bgr, gc_mask, None,
                    bgd_model, fgd_model,
                    n_iter, cv2.GC_INIT_WITH_MASK)
    except cv2.error:
        # Fallback: dùng rect nếu mask lỗi
        rect = (border, border, w - 2 * border, h - 2 * border)
        gc_mask[:] = cv2.GC_PR_BGD
        cv2.grabCut(img_bgr, gc_mask, rect,
                    bgd_model, fgd_model,
                    n_iter, cv2.GC_INIT_WITH_RECT)
 
    binary = np.where(
        (gc_mask == cv2.GC_FGD) | (gc_mask == cv2.GC_PR_FGD),
        255, 0
    ).astype(np.uint8)
 
    # Morphology cleanup
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
 
    masked = cv2.bitwise_and(img_bgr, img_bgr, mask=binary)
    return binary, masked

**TRÍCH XUẤT RATIO FEATURES**

In [35]:
def extract_ratio_features(img_bgr: np.ndarray,
                            mask: np.ndarray,
                            food_class: str = "") -> dict:
    """
    Trích xuất 5 ratio từ ảnh và mask.
    Tất cả đều nằm trong [0, 1] — dễ kết hợp.
 
    Ratios:
      R1 area_ratio      : diện tích foreground / tổng ảnh
                           → thức ăn chiếm bao nhiêu % khung hình
      R2 bbox_fill_ratio : diện tích foreground / diện tích bounding box
                           → thức ăn đặc hay thưa thớt trong vùng của nó
      R3 compactness     : 4π·area / perimeter²  (0=dài loằng ngoằng, 1=tròn)
                           → hình dạng tổng thể (pizza tròn vs salad vô định)
      R4 saturation_mean : độ bão hòa màu trung bình trong vùng thức ăn
                           → món nhiều màu sắc thường nhìn "nhiều" hơn
      R5 edge_density    : mật độ cạnh trong foreground
                           → texture phong phú (topping pizza) vs đơn giản
    """
    h, w = img_bgr.shape[:2]
    total_pixels = h * w
    fg_pixels = int(mask.sum() // 255)
 
    features = {
        "food_class"  : food_class,
        "total_pixels": total_pixels,
        "fg_pixels"   : fg_pixels,
    }
 
    # --- R1: Area ratio ---
    r1 = fg_pixels / total_pixels if total_pixels > 0 else 0.0
    features["R1_area_ratio"] = round(float(r1), 4)
 
    # --- R2: BBox fill ratio ---
    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    if contours:
        largest = max(contours, key=cv2.contourArea)
        x, y, bw, bh = cv2.boundingRect(largest)
        bbox_area = bw * bh
        r2 = fg_pixels / bbox_area if bbox_area > 0 else 0.0
        features["R2_bbox_fill"] = round(float(min(r2, 1.0)), 4)
        features["_bbox"] = (x, y, bw, bh)
 
        # --- R3: Compactness ---
        perimeter = cv2.arcLength(largest, closed=True)
        contour_area = cv2.contourArea(largest)
        if perimeter > 0:
            r3 = (4 * np.pi * contour_area) / (perimeter ** 2)
        else:
            r3 = 0.0
        features["R3_compactness"] = round(float(min(r3, 1.0)), 4)
    else:
        features["R2_bbox_fill"]  = 0.0
        features["R3_compactness"] = 0.0
        features["_bbox"] = (0, 0, w, h)
 
    # --- R4: Saturation mean trong foreground ---
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    fg_mask_bool = mask > 0
    if fg_mask_bool.any():
        sat_vals = hsv[:, :, 1][fg_mask_bool]
        r4 = float(sat_vals.mean()) / 255.0
    else:
        r4 = 0.0
    features["R4_saturation"] = round(r4, 4)
 
    # --- R5: Edge density trong foreground ---
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, threshold1=50, threshold2=150)
    if fg_pixels > 0:
        edges_in_fg = cv2.bitwise_and(edges, edges, mask=mask)
        r5 = float(edges_in_fg.sum() // 255) / fg_pixels
    else:
        r5 = 0.0
    features["R5_edge_density"] = round(float(min(r5, 1.0)), 4)
 
    return features

**TÍNH PORTION SCORE & GÁN NHÃN S/M/L**

In [36]:
# Trọng số cho từng ratio
# R1 (area) quan trọng nhất, R2 bổ trợ,
# R3-R5 dùng để phân biệt các trường hợp biên
WEIGHTS = {
    "R1_area_ratio": 0.50,
    "R2_bbox_fill" : 0.25,
    "R3_compactness": 0.05,
    "R4_saturation": 0.10,
    "R5_edge_density": 0.10,
}

In [37]:
def compute_weighted_score(features: dict) -> float:
    """
    Tính weighted_score ∈ [0, 1].
    Score cao → món ăn chiếm nhiều không gian → Large.
    """
    score = sum(
        WEIGHTS[k] * features.get(k, 0.0)
        for k in WEIGHTS
    )
    return round(float(score), 4)
 
 
def classify_portion(weighted_score: float,
                     food_class: str = "") -> str:
    """
    Gán nhãn Small / Medium / Large.
 
    Một số class cần hiệu chỉnh (per-class calibration):
      - Ảnh ramen/pho thường chụp từ trên, thức ăn chiếm gần hết frame
        → ngưỡng nên cao hơn một chút để tránh tất cả đều là Large
      - Pizza thường chụp cả chiếc → ngưỡng tương tự
    """
    # Per-class offset: điều chỉnh ngưỡng theo đặc thù class
    # Giá trị dương → nâng ngưỡng (class này hay bị ước lượng cao)
    # Giá trị âm  → hạ ngưỡng (class này hay bị ước lượng thấp)
    CLASS_OFFSET = {
        "ramen"        :  0.08,
        "pho"          :  0.08,
        "pizza"        :  0.06,
        "fried_rice"   :  0.05,
        "caesar_salad" : -0.05,
        "sushi"        : -0.03,
    }
    offset = CLASS_OFFSET.get(food_class, 0.0)
 
    t_small = THRESHOLDS["small"] + offset
    t_large = THRESHOLDS["large"] + offset
 
    if weighted_score < t_small:
        return "Small"
    elif weighted_score > t_large:
        return "Large"
    else:
        return "Medium"
 
 
def run_portion_pipeline(img_bgr: np.ndarray,
                         food_class: str = "") -> dict:
    """
    Chạy toàn bộ pipeline cho 1 ảnh.
    Trả về dict kết quả đầy đủ.
    """
    preprocessed = preprocess_image(img_bgr, size=512)
    mask, masked  = grabcut_segment(preprocessed)
    features      = extract_ratio_features(preprocessed, mask, food_class)
    score         = compute_weighted_score(features)
    portion       = classify_portion(score, food_class)
 
    return {
        **features,
        "weighted_score": score,
        "portion_label" : portion,
        "_mask"         : mask,
        "_masked_img"   : masked,
        "_preprocessed" : preprocessed,
    }

**GÁN NHÃN THỦ CÔNG (GROUND TRUTH)**

In [38]:
"""
Chiến lược gán nhãn:
  - Chọn ngẫu nhiên 30 ảnh mỗi class → 450 ảnh tổng
  - Hiển thị từng ảnh, người dùng nhập S / M / L
  - Lưu ra CSV để dùng ở Cell 6
 
Tiêu chí gán nhãn (nhất quán):
  Small  : Thức ăn chiếm < 1/3 khung hình, hoặc là 1 phần nhỏ của món
  Medium : Thức ăn chiếm 1/3 đến 2/3 khung hình
  Large  : Thức ăn chiếm > 2/3 khung hình, hoặc cả đĩa/tô đầy
"""
 
import random
 
N_LABEL_PER_CLASS = 30   # có thể giảm xuống 20 nếu ít thời gian

In [39]:
def collect_image_paths(classes: list,
                        n_per_class: int,
                        seed: int = 42) -> list[dict]:
    """Lấy ngẫu nhiên n ảnh mỗi class."""
    random.seed(seed)
    records = []
    for cls in classes:
        cls_dir = IMAGE_DIR / cls
        imgs = list(cls_dir.glob("*.jpg"))
        sampled = random.sample(imgs, min(n_per_class, len(imgs)))
        for p in sampled:
            records.append({"path": str(p), "food_class": cls})
    return records
 
 
def interactive_label(records: list[dict],
                      save_path: Path) -> pd.DataFrame:
    """
    Gán nhãn thủ công — chạy trong Kaggle notebook.
    Hiển thị ảnh + system prediction, người dùng xác nhận hoặc sửa.
    """
    results = []
    for i, rec in enumerate(records):
        img_bgr = cv2.imread(rec["path"])
        if img_bgr is None:
            continue
 
        result  = run_portion_pipeline(img_bgr, rec["food_class"])
        pred    = result["portion_label"]
        score   = result["weighted_score"]
 
        # Hiển thị ảnh và thông tin
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(cv2.cvtColor(result["_preprocessed"],
                                    cv2.COLOR_BGR2RGB))
        axes[0].set_title("Original")
        axes[0].axis("off")
 
        overlay = result["_preprocessed"].copy()
        overlay[result["_mask"] == 0] = (overlay[result["_mask"] == 0]
                                          * 0.3).astype(np.uint8)
        axes[1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
        axes[1].set_title(f"Segmented\nArea ratio: "
                          f"{result['R1_area_ratio']:.2%}")
        axes[1].axis("off")
 
        # Ratio bar chart
        ratio_names  = ["Area", "BBox fill", "Compact",
                        "Saturation", "Edge density"]
        ratio_values = [result["R1_area_ratio"],
                        result["R2_bbox_fill"],
                        result["R3_compactness"],
                        result["R4_saturation"],
                        result["R5_edge_density"]]
        bar_color = COLOR_MAP[pred]
        axes[2].barh(ratio_names, ratio_values, color=bar_color, alpha=0.8)
        axes[2].set_xlim(0, 1)
        axes[2].set_title(f"System: {pred}\n(score={score:.3f})")
        axes[2].axvline(0.5, color="gray", linestyle="--", alpha=0.5)
 
        fig.suptitle(f"[{i+1}/{len(records)}] {rec['food_class']}",
                     fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.show()
 
        # Nhập nhãn
        while True:
            user_input = input(
                f"  System dự đoán: [{pred}] | "
                "Nhãn của bạn (S/M/L hoặc Enter để đồng ý): "
            ).strip().upper()
            if user_input == "":
                label = pred
                break
            elif user_input in ("S", "M", "L"):
                label = {"S": "Small", "M": "Medium", "L": "Large"}[user_input]
                break
            else:
                print("  → Chỉ nhập S, M, L hoặc Enter")
 
        results.append({
            **{k: v for k, v in result.items()
               if not k.startswith("_")},
            "manual_label": label,
            "path": rec["path"],
        })
        plt.close("all")
 
    df = pd.DataFrame(results)
    df.to_csv(save_path, index=False)
    print(f"\n✓ Đã lưu {len(df)} nhãn → {save_path}")
    return df

**ĐÁNH GIÁ ĐỊNH LƯỢNG**

In [40]:
def evaluate_portion(df_labeled: pd.DataFrame) -> None:
    """
    Đánh giá hệ thống so với nhãn thủ công.
    In: accuracy, classification report, confusion matrix.
    """
    y_true = df_labeled["manual_label"].tolist()
    y_pred = df_labeled["portion_label"].tolist()
    labels = ["Small", "Medium", "Large"]
 
    acc = sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)
    print(f"\n{'='*50}")
    print(f"  PORTION ESTIMATION — ĐÁNH GIÁ TỔNG QUAN")
    print(f"{'='*50}")
    print(f"  Tổng ảnh đánh giá : {len(y_true)}")
    print(f"  Accuracy           : {acc:.1%}")
    print(f"\n{classification_report(y_true, y_pred,
                                     labels=labels,
                                     zero_division=0)}")
 
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=labels)
    disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
    axes[0].set_title("Confusion Matrix — Toàn bộ")
 
    # Per-class accuracy
    class_acc = {}
    for cls in CLASSES:
        sub = df_labeled[df_labeled["food_class"] == cls]
        if len(sub) > 0:
            ca = (sub["manual_label"] == sub["portion_label"]).mean()
            class_acc[cls] = ca
 
    sorted_cls = sorted(class_acc, key=class_acc.get)
    colors = ["#e74c3c" if v < 0.5 else
              "#f39c12" if v < 0.7 else
              "#27ae60" for v in class_acc.values()]
    sorted_colors = ["#e74c3c" if class_acc[c] < 0.5 else
                     "#f39c12" if class_acc[c] < 0.7 else
                     "#27ae60" for c in sorted_cls]
 
    axes[1].barh(sorted_cls,
                 [class_acc[c] for c in sorted_cls],
                 color=sorted_colors)
    axes[1].axvline(0.5, color="red", linestyle="--", alpha=0.7,
                    label="50%")
    axes[1].axvline(acc, color="navy", linestyle="-", alpha=0.7,
                    label=f"Overall {acc:.1%}")
    axes[1].set_xlim(0, 1)
    axes[1].set_xlabel("Accuracy")
    axes[1].set_title("Accuracy theo class")
    axes[1].legend()
 
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "evaluation.png", dpi=150,
                bbox_inches="tight")
    plt.show()
 
    # Phân tích lỗi: nhầm nhiều nhất
    errors = df_labeled[df_labeled["manual_label"] !=
                         df_labeled["portion_label"]]
    print(f"\n  Top lỗi phổ biến nhất:")
    error_pairs = (errors.groupby(["manual_label", "portion_label"])
                         .size()
                         .reset_index(name="count")
                         .sort_values("count", ascending=False))
    print(error_pairs.to_string(index=False))
 
    # Phân tích theo score
    _plot_score_distribution(df_labeled)

In [41]:
def _plot_score_distribution(df: pd.DataFrame) -> None:
    """
    Histogram weighted_score theo nhãn thật — cho thấy
    ngưỡng phân tách có hợp lý không.
    """
    fig, ax = plt.subplots(figsize=(9, 4))
    for label in ["Small", "Medium", "Large"]:
        sub = df[df["manual_label"] == label]["weighted_score"]
        ax.hist(sub, bins=20, alpha=0.6, label=label,
                color=COLOR_MAP[label])
    ax.axvline(THRESHOLDS["small"], color="gray",
               linestyle="--", label=f"T_small={THRESHOLDS['small']}")
    ax.axvline(THRESHOLDS["large"], color="black",
               linestyle="--", label=f"T_large={THRESHOLDS['large']}")
    ax.set_xlabel("Weighted Score")
    ax.set_ylabel("Số ảnh")
    ax.set_title("Phân bố Weighted Score theo nhãn thật")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "score_distribution.png",
                dpi=150, bbox_inches="tight")
    plt.show()

**VISUALISATION**

In [42]:
def visualize_samples(df: pd.DataFrame,
                      n_per_portion: int = 3) -> None:
    """
    Hiển thị n ảnh mỗi loại khẩu phần:
    ảnh gốc | mask overlay | ratio bars | nhãn S/M/L
    """
    fig_rows = []
    for portion in ["Small", "Medium", "Large"]:
        sub = df[df["portion_label"] == portion].sample(
            min(n_per_portion, len(df[df["portion_label"] == portion])),
            random_state=SEED
        )
        fig_rows.append(sub)
 
    samples = pd.concat(fig_rows).reset_index(drop=True)
    n = len(samples)
    fig, axes = plt.subplots(n, 3, figsize=(13, 4 * n))
    if n == 1:
        axes = [axes]
 
    for i, row in samples.iterrows():
        img_bgr = cv2.imread(row["path"])
        result  = run_portion_pipeline(img_bgr, row["food_class"])
 
        # Col 0: Original
        axes[i][0].imshow(cv2.cvtColor(result["_preprocessed"],
                                        cv2.COLOR_BGR2RGB))
        axes[i][0].set_title(f"{row['food_class']}", fontsize=10)
        axes[i][0].axis("off")
 
        # Col 1: Mask overlay
        overlay = result["_preprocessed"].copy()
        overlay[result["_mask"] == 0] = (
            overlay[result["_mask"] == 0] * 0.25).astype(np.uint8)
        axes[i][1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
        axes[i][1].set_title(
            f"Area ratio: {result['R1_area_ratio']:.2%}", fontsize=10)
        axes[i][1].axis("off")
 
        # Col 2: Ratio bars + portion badge
        ratio_names  = ["R1 Area", "R2 BBox fill",
                        "R3 Compact", "R4 Saturation", "R5 Edges"]
        ratio_values = [result["R1_area_ratio"],
                        result["R2_bbox_fill"],
                        result["R3_compactness"],
                        result["R4_saturation"],
                        result["R5_edge_density"]]
        portion_pred = result["portion_label"]
        color = COLOR_MAP[portion_pred]
        axes[i][2].barh(ratio_names, ratio_values,
                        color=color, alpha=0.8)
        axes[i][2].set_xlim(0, 1)
        axes[i][2].set_title(
            f"Portion: {portion_pred}  "
            f"(score={result['weighted_score']:.3f})", fontsize=10)
 
        # Vẽ ngưỡng
        for t, lbl in [(THRESHOLDS["small"], "T_small"),
                       (THRESHOLDS["large"], "T_large")]:
            axes[i][2].axvline(t, color="gray",
                               linestyle="--", alpha=0.6)
 
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "portion_samples.png",
                dpi=150, bbox_inches="tight")
    plt.show()
 

In [43]:
def plot_ratio_analysis(df: pd.DataFrame) -> None:
    """
    Boxplot 5 ratio theo portion label —
    giúp chọn trọng số WEIGHTS hợp lý hơn.
    """
    ratio_cols = ["R1_area_ratio", "R2_bbox_fill",
                  "R3_compactness", "R4_saturation", "R5_edge_density"]
    ratio_labels = ["R1 Area", "R2 BBox fill",
                    "R3 Compact", "R4 Saturation", "R5 Edges"]
 
    fig, axes = plt.subplots(1, 5, figsize=(18, 5))
    for ax, col, lbl in zip(axes, ratio_cols, ratio_labels):
        data_by_portion = [
            df[df["manual_label"] == p][col].dropna().values
            for p in ["Small", "Medium", "Large"]
        ]
        bp = ax.boxplot(data_by_portion,
                        labels=["S", "M", "L"],
                        patch_artist=True)
        for patch, color in zip(bp["boxes"],
                                 [COLOR_MAP["Small"],
                                  COLOR_MAP["Medium"],
                                  COLOR_MAP["Large"]]):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        ax.set_title(lbl, fontsize=11)
        ax.set_ylim(0, 1)
        ax.set_ylabel("Value")
 
    fig.suptitle("Phân bố từng Ratio theo nhãn khẩu phần",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "ratio_boxplots.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    print("\nTừ boxplot: ratio nào có hộp tách biệt tốt nhất giữa S/M/L")
    print("→ tăng trọng số WEIGHTS của ratio đó để cải thiện accuracy.")

**LƯU KẾT QUẢ & CHẠY TOÀN BỘ PIPELINE**

In [44]:
def batch_run(classes: list,
              n_per_class: int = N_SAMPLE_PER_CLASS) -> pd.DataFrame:
    """
    Chạy pipeline trên toàn bộ ảnh (không cần nhãn thủ công).
    Dùng để quan sát phân bố prediction trước khi gán nhãn.
    """
    records = collect_image_paths(classes, n_per_class, seed=SEED)
    results = []
    for rec in tqdm(records, desc="Running portion pipeline"):
        img_bgr = cv2.imread(rec["path"])
        if img_bgr is None:
            continue
        result = run_portion_pipeline(img_bgr, rec["food_class"])
        results.append({
            k: v for k, v in result.items()
            if not k.startswith("_")
        })
 
    df = pd.DataFrame(results)
    df.to_csv(OUTPUT_DIR / "portion_predictions.csv", index=False)
    print(f"✓ Đã xử lý {len(df)} ảnh")
    print("\nPhân bố portion labels:")
    print(df["portion_label"].value_counts())
    print("\nPhân bố theo class:")
    print(df.groupby("food_class")["portion_label"]
            .value_counts()
            .unstack(fill_value=0))
    return df

**Chạy pipeline**

In [ ]:
if __name__ == "__main__":
 
    # BƯỚC 1: Chạy batch để xem phân bố tổng thể
    print("=" * 55)
    print("BƯỚC 1: BATCH RUN — xem phân bố prediction")
    print("=" * 55)
    df_batch = batch_run(CLASSES, n_per_class=N_SAMPLE_PER_CLASS)
 
    # BƯỚC 2: Gán nhãn thủ công (chỉ cần 1 lần, lưu CSV)
    label_csv = OUTPUT_DIR / "manual_labels.csv"
    if not label_csv.exists():
        print("\n" + "=" * 55)
        print("BƯỚC 2: GÁN NHÃN THỦ CÔNG")
        print("=" * 55)
        records_to_label = collect_image_paths(
            CLASSES, N_LABEL_PER_CLASS, seed=SEED
        )
        df_labeled = interactive_label(records_to_label, label_csv)
    else:
        print(f"\n✓ Đã có file nhãn: {label_csv}")
        df_labeled = pd.read_csv(label_csv)
 
    # BƯỚC 3: Đánh giá
    print("\n" + "=" * 55)
    print("BƯỚC 3: ĐÁNH GIÁ ĐỊNH LƯỢNG")
    print("=" * 55)
    evaluate_portion(df_labeled)
 
    # BƯỚC 4: Visualisation
    print("\n" + "=" * 55)
    print("BƯỚC 4: VISUALISATION")
    print("=" * 55)
    visualize_samples(df_labeled, n_per_portion=3)
    plot_ratio_analysis(df_labeled)
 
    print("\n✓ Hoàn thành! Kết quả lưu tại:", OUTPUT_DIR)
 

BƯỚC 1: BATCH RUN — xem phân bố prediction


Running portion pipeline:  50%|████▉     | 743/1500 [34:24<21:23,  1.70s/it]  